# Step 1 — Understand & Clean the Telco Customer Churn Dataset

**Input:** `WA_Fn-UseC_-Telco-Customer-Churn.csv` (raw, IBM Telco Customer Churn dataset)
**Output:** `data/cleaned_telco.csv`

7,043 customers, one row each. This notebook profiles the raw data, fixes the
one real data-quality issue (`TotalCharges` blank for brand-new customers),
and adds the derived columns the rest of the pipeline depends on.

In [1]:
import os
import sys


def find_project_root(marker="requirements.txt"):
    path = os.getcwd()
    for _ in range(5):
        if os.path.exists(os.path.join(path, marker)):
            return path
        path = os.path.dirname(path)
    raise FileNotFoundError(f"Could not locate project root (looked for {marker})")


PROJECT_ROOT = find_project_root()
sys.path.insert(0, os.path.join(PROJECT_ROOT, "scripts"))
print(f"PROJECT_ROOT = {PROJECT_ROOT}")

PROJECT_ROOT = C:\Users\DEVKUMAR\OneDrive\Desktop\DA_Project


In [2]:
import numpy as np
import pandas as pd

RAW_PATH = os.path.join(PROJECT_ROOT, "WA_Fn-UseC_-Telco-Customer-Churn.csv")
OUT_PATH = os.path.join(PROJECT_ROOT, "data", "cleaned_telco.csv")

ADDON_SERVICES = [
    "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies",
]

## Load and take a first look

In [3]:
df = pd.read_csv(RAW_PATH)
n_raw = len(df)
print(f"Loaded {n_raw} rows, {df.shape[1]} columns")
df.head()

Loaded 7043 rows, 21 columns


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Null counts before cleaning

`TotalCharges` is read as a string because 11 rows are blank.

In [4]:
print("Null counts (raw):")
print(df.isnull().sum()[df.isnull().sum() > 0])

Null counts (raw):
Series([], dtype: int64)


## Fix `TotalCharges`

The blanks are exactly the brand-new (`tenure == 0`) customers who haven't been billed yet — set them to 0 rather than dropping.

In [5]:
df["TotalCharges"] = df["TotalCharges"].replace(" ", np.nan).astype(float)
blank_total_mask = df["TotalCharges"].isnull()
print(f"Blank TotalCharges: {blank_total_mask.sum()} rows, all tenure==0? "
      f"{(df.loc[blank_total_mask, 'tenure'] == 0).all()}")
df.loc[blank_total_mask, "TotalCharges"] = 0.0

Blank TotalCharges: 11 rows, all tenure==0? True


## Type fixes

In [6]:
df["SeniorCitizen"] = df["SeniorCitizen"].map({0: "No", 1: "Yes"})
df["tenure"] = df["tenure"].astype(int)
df["MonthlyCharges"] = df["MonthlyCharges"].astype(float)
df[["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges"]].dtypes

SeniorCitizen         str
tenure              int64
MonthlyCharges    float64
TotalCharges      float64
dtype: object

## Sanity cross-check

`TotalCharges` should roughly equal `tenure * MonthlyCharges`.

In [7]:
df["AvgMonthlySpend"] = df["TotalCharges"] / df["tenure"].replace(0, 1)
diff = (df["AvgMonthlySpend"] - df["MonthlyCharges"]).abs()
print(f"AvgMonthlySpend vs MonthlyCharges - median abs diff: {diff.median():.2f}, "
      f"95th pct: {diff.quantile(0.95):.2f}")

AvgMonthlySpend vs MonthlyCharges - median abs diff: 1.16, 95th pct: 5.53


## Derived features

`NumServices` (add-on count) and `ChurnFlag` (0/1) are used throughout the rest of the pipeline.

In [8]:
df["NumServices"] = (df[ADDON_SERVICES] == "Yes").sum(axis=1)
df["ChurnFlag"] = (df["Churn"] == "Yes").astype(int)
df[["NumServices", "ChurnFlag"]].describe()

,NumServices,ChurnFlag
count,7043.000000,7043.000000
mean,2.037910,0.265370
std,1.847682,0.441561
min,0.000000,0.000000
25%,0.000000,0.000000
50%,2.000000,0.000000
75%,3.000000,1.000000
max,6.000000,1.000000


## Data quality checks

In [9]:
dup_ids = df["customerID"].duplicated().sum()
print(f"Duplicate customerIDs: {dup_ids}")

Duplicate customerIDs: 0


In [10]:
for col in ["Contract", "PaymentMethod", "InternetService"] + ADDON_SERVICES:
    print(f"\n{col} value counts:")
    print(df[col].value_counts())


Contract value counts:
Contract
Month-to-month    3875
Two year          1695
One year          1473
Name: count, dtype: int64

PaymentMethod value counts:
PaymentMethod
Electronic check             2365
Mailed check                 1612
Bank transfer (automatic)    1544
Credit card (automatic)      1522
Name: count, dtype: int64

InternetService value counts:
InternetService
Fiber optic    3096
DSL            2421
No             1526
Name: count, dtype: int64

OnlineSecurity value counts:
OnlineSecurity
No                     3498
Yes                    2019
No internet service    1526
Name: count, dtype: int64

OnlineBackup value counts:
OnlineBackup
No                     3088
Yes                    2429
No internet service    1526
Name: count, dtype: int64

DeviceProtection value counts:
DeviceProtection
No                     3095
Yes                    2422
No internet service    1526
Name: count, dtype: int64

TechSupport value counts:
TechSupport
No                     3473
Ye

In [11]:
print(f"Overall churn rate: {df['ChurnFlag'].mean():.4f} ({df['ChurnFlag'].sum()} / {len(df)})")

Overall churn rate: 0.2654 (1869 / 7043)


## Save cleaned data

In [12]:
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
df.to_csv(OUT_PATH, index=False)
print(f"Wrote {len(df)} rows to {OUT_PATH}")

assert len(df) == n_raw, "Row count changed during cleaning"
assert df["TotalCharges"].isnull().sum() == 0
assert df["tenure"].isnull().sum() == 0
print("All assertions passed.")

Wrote 7043 rows to C:\Users\DEVKUMAR\OneDrive\Desktop\DA_Project\data\cleaned_telco.csv
All assertions passed.
